<a href="https://colab.research.google.com/github/githbsingh/pyspark/blob/main/Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Reading File from HDFS to RDD
- sc.textFile -> TextFile
- sc.sequenceFile -> Sequence File
#### Writing data into HDFS from RDD
- saveAsTextFile
- saveAsSequnceFile

In [ ]:
# File location and type
file_location = "/FileStore/tables/part_00000.csv"
file_type = "csv"

# CSV options
infer_schema = "false"
first_row_is_header = "false"
delimiter = ","

# The applied options are for CSV files. For other file types, these will be ignored.
df = spark.read.format(file_type) \
  .option("inferSchema", infer_schema) \
  .option("header", first_row_is_header) \
  .option("sep", delimiter) \
  .load(file_location)
df.head()

Out[1]: Row(_c0='1', _c1='2013-07-25 00:00:00.0', _c2='11599', _c3='CLOSED')

In [ ]:
print(type(df))
order = df.rdd.map(lambda row : ",".join(str(x) for x in row))
print(type(order))
for i in order.take(5) : print(i)

<class 'pyspark.sql.dataframe.DataFrame'>
<class 'pyspark.rdd.PipelinedRDD'>
1,2013-07-25 00:00:00.0,11599,CLOSED
2,2013-07-25 00:00:00.0,256,PENDING_PAYMENT
3,2013-07-25 00:00:00.0,12111,COMPLETE
4,2013-07-25 00:00:00.0,8827,CLOSED
5,2013-07-25 00:00:00.0,11318,COMPLETE


#### Get orders for the month of july and august

In [ ]:
julOrd = order.filter(lambda x: (x.split(',')[1].split('-')[1])=='07')
for i in julOrd.take(5) : print(i)

1,2013-07-25 00:00:00.0,11599,CLOSED
2,2013-07-25 00:00:00.0,256,PENDING_PAYMENT
3,2013-07-25 00:00:00.0,12111,COMPLETE
4,2013-07-25 00:00:00.0,8827,CLOSED
5,2013-07-25 00:00:00.0,11318,COMPLETE


In [ ]:
augOrd = order.filter(lambda x: (x.split(',')[1].split('-')[1])=='08')
for i in augOrd.take(5) : print(i)

1297,2013-08-01 00:00:00.0,11607,COMPLETE
1298,2013-08-01 00:00:00.0,5105,CLOSED
1299,2013-08-01 00:00:00.0,7802,COMPLETE
1300,2013-08-01 00:00:00.0,553,PENDING_PAYMENT
1301,2013-08-01 00:00:00.0,1604,PENDING_PAYMENT


In [ ]:
julAugOrders = julOrd.union(augOrd).distinct()

In [ ]:
julAugOrders.getNumPartitions()

Out[6]: 2

In [ ]:
julAugOrders.repartition(5).saveAsTextFile('/Shared/Pyspark/julAugOrd',compressionCodecClass='org.apache.hadoop.io.compress.BZip2Codec')

---------------------------------------------------------------------------
Py4JJavaError                             Traceback (most recent call last)
File <command-518376604166644>:1
----> 1 julAugOrders.repartition(5).saveAsTextFile('/Shared/Pyspark/julAugOrd',compressionCodecClass='org.apache.hadoop.io.compress.BZip2Codec')

File /databricks/spark/python/pyspark/instrumentation_utils.py:48, in _wrap_function.<locals>.wrapper(*args, **kwargs)
     46 start = time.perf_counter()
     47 try:
---> 48     res = func(*args, **kwargs)
     49     logger.log_success(
     50         module_name, class_name, function_name, time.perf_counter() - start, signature
     51     )
     52     return res

File /databricks/spark/python/pyspark/rdd.py:3430, in RDD.saveAsTextFile(self, path, compressionCodecClass)
   3427 # We must pull this out into two separate if branches because Py4J cannot translate from
   3428 # Python None --> Java null
   3429 if compressionCodecClass:
-> 3430     self.ctx.

In [ ]:
for i in julAugOrders.take(500): print(i)

6,2013-07-25 00:00:00.0,7130,COMPLETE
9,2013-07-25 00:00:00.0,5657,PENDING_PAYMENT
10,2013-07-25 00:00:00.0,5648,PENDING_PAYMENT
11,2013-07-25 00:00:00.0,918,PAYMENT_REVIEW
12,2013-07-25 00:00:00.0,1837,CLOSED
13,2013-07-25 00:00:00.0,9149,PENDING_PAYMENT
14,2013-07-25 00:00:00.0,9842,PROCESSING
17,2013-07-25 00:00:00.0,2667,COMPLETE
19,2013-07-25 00:00:00.0,9488,PENDING_PAYMENT
20,2013-07-25 00:00:00.0,9198,PROCESSING
23,2013-07-25 00:00:00.0,4367,PENDING_PAYMENT
30,2013-07-25 00:00:00.0,10039,PENDING_PAYMENT
31,2013-07-25 00:00:00.0,6983,PAYMENT_REVIEW
34,2013-07-25 00:00:00.0,4189,PROCESSING
35,2013-07-25 00:00:00.0,4840,COMPLETE
36,2013-07-25 00:00:00.0,5649,PENDING
37,2013-07-25 00:00:00.0,5863,CLOSED
39,2013-07-25 00:00:00.0,8214,PENDING
41,2013-07-25 00:00:00.0,8136,PENDING_PAYMENT
45,2013-07-25 00:00:00.0,2636,COMPLETE
53,2013-07-25 00:00:00.0,4701,PROCESSING
55,2013-07-25 00:00:00.0,2052,PENDING
59,2013-07-25 00:00:00.0,11644,PENDING_PAYMENT
62,2013-07-25 00:00:00.0,9111,CLOSE

In [ ]:
julAugPair = julAugOrders.map(lambda x: (int(x.split(',')[0]),x))

In [ ]:
for i in julAugPair.take(5) : print(i)

(6, '6,2013-07-25 00:00:00.0,7130,COMPLETE')
(9, '9,2013-07-25 00:00:00.0,5657,PENDING_PAYMENT')
(10, '10,2013-07-25 00:00:00.0,5648,PENDING_PAYMENT')
(11, '11,2013-07-25 00:00:00.0,918,PAYMENT_REVIEW')
(12, '12,2013-07-25 00:00:00.0,1837,CLOSED')


In [ ]:
julAugPair.saveAsSequenceFile('/Shared/Pyspark/julAugOrdSeq')

In [ ]:
rdd = sc.sequenceFile('/Shared/Pyspark/julAugOrdSeq')

In [ ]:
for i in rdd.take(5) : print(i)

(6, '6,2013-07-25 00:00:00.0,7130,COMPLETE')
(9, '9,2013-07-25 00:00:00.0,5657,PENDING_PAYMENT')
(10, '10,2013-07-25 00:00:00.0,5648,PENDING_PAYMENT')
(11, '11,2013-07-25 00:00:00.0,918,PAYMENT_REVIEW')
(12, '12,2013-07-25 00:00:00.0,1837,CLOSED')


In [ ]:
# Create pair using None as key
julAugPair = julAugOrders.map(lambda x: (None,x))

In [ ]:
for i in julAugPair.take(5) : print(i)

(None, '6,2013-07-25 00:00:00.0,7130,COMPLETE')
(None, '9,2013-07-25 00:00:00.0,5657,PENDING_PAYMENT')
(None, '10,2013-07-25 00:00:00.0,5648,PENDING_PAYMENT')
(None, '11,2013-07-25 00:00:00.0,918,PAYMENT_REVIEW')
(None, '12,2013-07-25 00:00:00.0,1837,CLOSED')


In [ ]:
julAugPair.coalesce(1).saveAsSequenceFile('/Shared/Pyspark/julAugOrdSeq_1')


In [ ]:
rdd = sc.sequenceFile('/Shared/Pyspark/julAugOrdSeq_1')

In [ ]:
for i in rdd.take(10): print(i)

(None, '6,2013-07-25 00:00:00.0,7130,COMPLETE')
(None, '9,2013-07-25 00:00:00.0,5657,PENDING_PAYMENT')
(None, '10,2013-07-25 00:00:00.0,5648,PENDING_PAYMENT')
(None, '11,2013-07-25 00:00:00.0,918,PAYMENT_REVIEW')
(None, '12,2013-07-25 00:00:00.0,1837,CLOSED')
(None, '13,2013-07-25 00:00:00.0,9149,PENDING_PAYMENT')
(None, '14,2013-07-25 00:00:00.0,9842,PROCESSING')
(None, '17,2013-07-25 00:00:00.0,2667,COMPLETE')
(None, '19,2013-07-25 00:00:00.0,9488,PENDING_PAYMENT')
(None, '20,2013-07-25 00:00:00.0,9198,PROCESSING')
